# Volumes

A container's writable layer is ephemeral — it vanishes when the container is removed. For anything that needs to outlive a container — database files, user uploads, logs, shared configuration — you need persistent storage.

Docker provides three storage mechanisms: **volumes**, **bind mounts**, and **tmpfs**. Volumes are the recommended approach for persistent data. They are managed by Docker, portable, and work correctly across platforms.

Topics:
- Why the writable layer is not enough
- Named volumes vs anonymous volumes
- Creating, inspecting, and removing volumes
- Mounting volumes with `-v` and `--mount`
- Sharing volumes between containers
- Volume drivers and remote storage
- Backing up and restoring volume data
- Volumes in Dockerfile: the `VOLUME` instruction

## 1. The Problem with the Writable Layer

Every container has a thin writable layer on top of its image layers. Writes go here. When you remove the container, this layer is deleted — all data written to it is gone.

```
docker run postgres          # database files written to writable layer
docker rm postgres           # writable layer deleted — all data lost
```

Beyond data loss, the writable layer has other drawbacks:
- It uses a **copy-on-write** storage driver (overlay2) which adds overhead for write-heavy workloads
- Data cannot be **shared** between containers
- Data cannot be **inspected or backed up** easily from the host
- Data does not **migrate** cleanly when you replace a container with a newer image

Volumes solve all of these.

## 2. How Volumes Work

A Docker volume is a directory managed by the Docker daemon, stored on the host at `/var/lib/docker/volumes/<name>/_data` (on Linux). When you mount a volume into a container, Docker bind-mounts that directory into the container's filesystem — bypassing the copy-on-write overlay entirely.

```
Host filesystem                      Container filesystem
/var/lib/docker/volumes/
  pgdata/
    _data/                ──────────►  /var/lib/postgresql/data/
      base/
      pg_wal/
      PG_VERSION
```

Key properties:
- **Independent lifecycle** — volumes exist until you explicitly delete them, regardless of which containers use them
- **Portable** — volumes can be moved between containers by name
- **No copy-on-write overhead** — writes go directly to the host filesystem
- **Cross-platform** — work on Linux, macOS (via the Docker Desktop VM), and Windows

## 3. Named Volumes vs Anonymous Volumes

### Named volume
You give the volume an explicit name. Docker creates it the first time it's referenced. Use named volumes for anything you care about.

```bash
docker run -v pgdata:/var/lib/postgresql/data postgres:16
#              ^^^^^^ volume name
```

### Anonymous volume
No name is given — Docker generates a random UUID as the name. Created when you use `-v /container/path` without a name, or when a `VOLUME` instruction exists in the Dockerfile and no volume is explicitly mounted.

```bash
docker run -v /var/lib/postgresql/data postgres:16
#              no name — Docker generates a random UUID
```

Anonymous volumes are hard to manage — you can't easily reference them by name. They accumulate as orphaned volumes and are removed by `docker volume prune`. Prefer named volumes.

## 4. Managing Volumes

In [ ]:
# Create a named volume explicitly
!docker volume create pgdata

# List all volumes
!docker volume ls

# Inspect a volume — shows mount point, driver, labels
!docker volume inspect pgdata

In [ ]:
# Write data to the volume using one container
!docker run --rm -v pgdata:/data alpine \
    sh -c 'echo "persistent data" > /data/hello.txt && echo "Written to volume"'

# Read it back using a completely different container — same volume
!docker run --rm -v pgdata:/data alpine cat /data/hello.txt

In [ ]:
# The data survives container removal
!docker run --rm -v pgdata:/data alpine sh -c 'echo "second write" >> /data/hello.txt'
!docker run --rm -v pgdata:/data alpine cat /data/hello.txt
# Both writes are there — neither container existed when we read

In [ ]:
# Remove a specific volume (must not be in use by any container)
!docker volume rm pgdata

# Remove all unused volumes (no running or stopped container references them)
# docker volume prune -f

print("Volume removed")

## 5. `-v` vs `--mount`

There are two syntaxes for mounting volumes. Both work; `--mount` is more explicit and preferred for clarity.

### Short syntax: `-v`

```bash
-v volume_name:/container/path          # named volume
-v /host/path:/container/path           # bind mount (if starts with /)
-v /container/path                      # anonymous volume
-v volume_name:/container/path:ro       # read-only
```

### Long syntax: `--mount`

```bash
--mount type=volume,source=pgdata,target=/var/lib/postgresql/data
--mount type=volume,source=pgdata,target=/data,readonly
--mount type=bind,source=/host/path,target=/container/path
--mount type=tmpfs,target=/tmp
```

The `type=` field makes the mount type explicit — no ambiguity between a volume name and an absolute path. For volumes, if `source` doesn't exist, Docker creates it automatically.

In [ ]:
# Equivalent commands — same result

# Short syntax
!docker run --rm -v appdata:/app/data alpine ls /app/data

# Long syntax — more explicit, less ambiguous
!docker run --rm \
    --mount type=volume,source=appdata,target=/app/data \
    alpine ls /app/data

!docker volume rm appdata

In [ ]:
# Read-only volume mount — container can read but not write
!docker volume create readonly-demo
!docker run --rm -v readonly-demo:/data alpine sh -c 'echo "seed data" > /data/config.txt'

# Mount as read-only
!docker run --rm -v readonly-demo:/data:ro alpine sh -c '
    echo "Can read:"
    cat /data/config.txt
    echo "Trying to write..."
    echo "attempt" > /data/config.txt 2>&1 || echo "Write blocked — read-only mount"
'

!docker volume rm readonly-demo

## 6. Sharing a Volume Between Containers

Multiple containers can mount the same volume simultaneously. This is how you share data between services — a writer and a reader, a primary and a sidecar, a web server and a log shipper.

In [ ]:
import time

# Writer container: appends a timestamp to a log file every second
!docker run -d --name log-writer -v shared-logs:/logs \
    alpine sh -c 'while true; do date >> /logs/app.log; sleep 1; done'

time.sleep(3)   # let writer produce some entries

# Reader container: reads from the same volume
!docker run --rm -v shared-logs:/logs alpine \
    sh -c 'echo "Lines written:"; wc -l /logs/app.log; echo "Last 3:"; tail -3 /logs/app.log'

!docker rm -f log-writer
!docker volume rm shared-logs

**Concurrent writes:** Docker does not provide locking. If two containers write to the same file simultaneously, you can get corrupted data. Coordinate writes at the application level — use a database, a message queue, or file locking.

## 7. Pre-Populating a Volume

When you mount a **named volume** into a container and the volume is **empty**, Docker copies the contents of the container's image at that path into the volume. This is a one-time initialisation.

```dockerfile
FROM nginx:alpine
# /usr/share/nginx/html contains the default index.html
```

```bash
# First run: empty volume 'webroot' → populated with Nginx's default files
docker run -v webroot:/usr/share/nginx/html nginx:alpine

# Subsequent runs: volume already has data — image contents are NOT copied again
docker run -v webroot:/usr/share/nginx/html nginx:alpine
```

This behaviour only applies to named volumes. It does **not** apply to bind mounts — if the host directory is empty, the container sees an empty directory.

In [ ]:
# New empty volume mounted at /usr/share/nginx/html
# Docker seeds it with the image's contents at that path
!docker run --rm -v webroot:/usr/share/nginx/html nginx:alpine \
    ls /usr/share/nginx/html

# The files are now on the host volume
!docker run --rm -v webroot:/data alpine ls /data

!docker volume rm webroot

## 8. Backing Up and Restoring Volume Data

Since volume data lives at `/var/lib/docker/volumes/` on the host, you can back it up by running a helper container that tars the volume and writes to a host path.

In [ ]:
import os

# Create a volume with some data
!docker volume create backup-demo
!docker run --rm -v backup-demo:/data alpine \
    sh -c 'echo "important data" > /data/db.txt && echo "config" > /data/config.json'

os.makedirs("/tmp/vol-backup", exist_ok=True)

# BACKUP: mount volume + host backup dir, tar the volume to the host
!docker run --rm \
    -v backup-demo:/source:ro \
    -v /tmp/vol-backup:/backup \
    alpine tar czf /backup/backup-demo.tar.gz -C /source .

!ls -lh /tmp/vol-backup/
!tar tzf /tmp/vol-backup/backup-demo.tar.gz

In [ ]:
# RESTORE: create a new volume, extract the backup into it
!docker volume create restored-demo

!docker run --rm \
    -v restored-demo:/target \
    -v /tmp/vol-backup:/backup:ro \
    alpine tar xzf /backup/backup-demo.tar.gz -C /target

# Verify the restore
!docker run --rm -v restored-demo:/data alpine ls /data
!docker run --rm -v restored-demo:/data alpine cat /data/db.txt

# Clean up
!docker volume rm backup-demo restored-demo
!rm -rf /tmp/vol-backup

## 9. The `VOLUME` Instruction in Dockerfiles

```dockerfile
VOLUME /var/lib/postgresql/data
```

The `VOLUME` instruction declares that the specified path should be externally mounted. If no volume is explicitly provided at `docker run`, Docker creates an anonymous volume at that path.

**Implications:**
- Any `RUN` instruction after `VOLUME` cannot write to that path — the volume mount overrides the filesystem at that point
- Writes to that path in subsequent Dockerfile instructions are discarded
- It is a signal to operators: "this path contains important data — mount a volume here"

**In practice:** Most custom Dockerfiles don't need `VOLUME` — it's mainly in official database images (postgres, mysql, redis) where omitting a volume mount would silently lose data. When you build your own images, document the expected volume mounts with `VOLUME` or in the README.

```bash
# See which paths are declared as volumes in an image
docker inspect postgres:16 --format '{{json .Config.Volumes}}'
```

In [ ]:
# Inspect what volumes the postgres image declares
import subprocess, json

!docker pull --quiet postgres:16

raw = subprocess.check_output(["docker", "inspect", "postgres:16", "--format", "{{json .Config.Volumes}}"])
volumes = json.loads(raw)
print("Volumes declared by postgres:16:")
for path in (volumes or {}):
    print(f"  {path}")

## 10. Volume Drivers

The default volume driver is `local` — data lives on the host's disk. Docker supports third-party volume drivers that store data elsewhere:

| Driver | Storage backend |
|--------|----------------|
| `local` | Host disk (default) |
| `nfs` | NFS share (via local driver options) |
| `convoy` | Block storage (EBS, etc.) |
| `rexray` | Multi-cloud block/object storage |
| `azure-file` | Azure File Share |

```bash
# Create a volume backed by an NFS share
docker volume create \
    --driver local \
    --opt type=nfs \
    --opt o=addr=192.168.1.1,rw \
    --opt device=:/exports/data \
    nfs-data
```

Volume drivers are how you get data to persist across host failures in orchestrated environments (Kubernetes uses a similar concept via Persistent Volumes and StorageClasses).

## Summary

- A container's **writable layer** is deleted when the container is removed. Volumes provide persistent storage that outlives containers.
- **Named volumes** are the recommended approach — managed by Docker, stored at `/var/lib/docker/volumes/<name>/_data`, and referenced by name across containers.
- **Anonymous volumes** have a random UUID name, are hard to manage, and should be avoided in favour of named volumes.
- Use `--mount type=volume,source=name,target=/path` (explicit) or `-v name:/path` (short form) to attach a volume.
- Add `:ro` / `readonly` for a **read-only mount** — the container can read but not write.
- **Sharing:** multiple containers can mount the same volume. Docker provides no locking — coordinate writes at the application level.
- **Pre-population:** mounting an empty named volume into a path that has content in the image seeds the volume with that content on first use.
- **Backup:** run a helper container that mounts the volume and tars it to a host-mounted directory. Restore by extracting into a new volume.
- The `VOLUME` instruction in a Dockerfile declares paths that should be externally mounted — mainly used by official database images.
- **Volume drivers** allow backends beyond local disk: NFS, cloud block storage, and more.